# Implements a random Gx/Gx/c queue generator
### Authors: Rafael Andrade & Eriky Silva & Frederico Cruz
---

Required packages

In [1]:
import random as rnd
import heapq as hq
import pandas as pd
import numpy as np
import math

import pickle
from datetime import date

## Queue generator class
---

In [2]:
class Queue:
    def __init__(self, farr, fdep, tmax, nserv=1, farr_batch=lambda: 1, fdep_batch=lambda: 1):
        self.elements = []
        self.nserv = nserv
        self.farr = farr
        self.fdep = fdep
        self.farr_batch = farr_batch
        self.fdep_batch = fdep_batch
        self.tmax = tmax

        self.queue = 0
        self.busyserv = 0
        self.time = 0
        self.events = []

        self.log_time = []
        self.log_event = []
        self.log_queue = []
        self.log_busy = []
        self.total_arrivals = 0
        self.total_departures = 0
        
        self.schedule_arr()


    def schedule_arr(self):
        arr_time = self.time + self.farr()
        hq.heappush(self.events, (arr_time, 'arr'))


    def schedule_dep(self):
        dep_time = self.time + self.fdep()
        hq.heappush(self.events, (dep_time, 'dep'))


    def try_start_service(self):
        while self.busyserv < self.nserv and self.queue > 0:
            batch = min(self.fdep_batch(), self.queue)
            self.queue -= batch
            self.total_departures += batch
            self.busyserv += 1
            self.schedule_dep()

            
    def process_arr(self):
        self.schedule_arr()
        batch = self.farr_batch()
        self.queue += batch
        self.total_arrivals += batch
        self.try_start_service()


    def process_dep(self):
        self.busyserv -= 1
        self.try_start_service()


    def log_state(self, event):
        self.log_time.append(self.time)
        self.log_event.append(event)
        self.log_queue.append(self.queue)
        self.log_busy.append(self.busyserv)


    def run(self):
        while self.events and self.time < self.tmax:
            self.time, ev = hq.heappop(self.events)

            if ev == 'arr':
                self.process_arr()
            else:
                self.process_dep()

            self.log_state(ev)

    
    def get_metrics(self):
        if len(self.log_time) < 2:
            print("Not enough data")
            return

        total_time = self.log_time[-1]

        area_q = 0.0
        area_b = 0.0
        area_s = 0.0

        for i in range(1, len(self.log_time)):
            dt = self.log_time[i] - self.log_time[i-1]

            q = self.log_queue[i-1]
            b = self.log_busy[i-1]
            s = q + b

            area_q += q * dt
            area_b += b * dt
            area_s += s * dt

        avg_q = area_q / total_time
        avg_b = area_b / total_time
        avg_s = area_s / total_time

        n_arr = self.total_arrivals
        n_dep = self.total_departures

        utilization = avg_b / self.nserv
        lmbda_eff = n_arr / total_time
        throughput = n_dep / total_time

        w = avg_s / lmbda_eff if lmbda_eff > 0 else 0
        wq = avg_q / lmbda_eff if lmbda_eff > 0 else 0

        return {
        "rho": utilization,
        "L": avg_s,
        "Lq": avg_q,
        "W": w,
        "Wq": wq,
        "avg_in_service": avg_b,
        "throughput": throughput,
        "total_time": total_time,
        "arrivals": n_arr,
        "departures": n_dep
    }

    def print_metrics(self):
        metrics = self.get_metrics()
        print("=== Queue Metrics ===")
        print(f"Rho: {metrics['rho']:.4f}")
        print(f"L: {metrics['L']:.4f}")
        print(f"Lq: {metrics['Lq']:.4f}")
        print(f"W: {metrics['W']:.4f}")
        print(f"Wq: {metrics['Wq']:.4f}")
        print(f"Avg in service: {metrics['avg_in_service']:.4f}")
        print(f"Throughput: {metrics['throughput']:.4f}")
        print(f"Total time: {metrics['total_time']:.4f}")
        print(f"Arrivals: {metrics['arrivals']}")
        print(f"Departures: {metrics['departures']}")

    
    def print_log(self):
        return pd.DataFrame({
            'time': self.log_time,
            'event': self.log_event,
            'queue': self.log_queue,
            'busy': self.log_busy
        })


## Simulating queues
---

#### Monte Carlo functions

In [3]:
def mc_queue(farr, fdep, tmax, nserv, farr_batch, fdep_batch, rep_mc=100):
    np.random.seed(2026)
    res = []
    for _ in range(rep_mc):
        q = Queue(farr, fdep, tmax, nserv, farr_batch, fdep_batch)
        q.run()
        m = q.get_metrics()
        res.append([m['Wq'], m['Lq'], m['rho']])
    
    df = pd.DataFrame(res, columns=['Wq', 'Lq', 'Rho'])
    return df.mean().add_suffix('_mean').to_dict() | df.var().add_suffix('_var').to_dict()


def tab_mc_queue(lambdas, mus, tmaxs, farr, fdep, nserv, farr_batch, fdep_batch, rep_mc=100):
    tab = []
    total = len(tmaxs) * len(lambdas)
    k = 0
    for lmb, mu in zip(lambdas, mus):
        for tmax in tmaxs:
            k += 1
            stats = mc_queue(farr(lmb), fdep(mu), tmax, nserv, farr_batch, fdep_batch, rep_mc)
            tab.append({'lambda': lmb, 'mu': mu, 'tmax': tmax} | stats)
            print(f"\r{100*k/total:.1f}%", end="")
    return pd.DataFrame(tab)

### Parameters

In [4]:
arr_batchs = np.array([1, 2, 3])
dep_batchs = np.array([1, 2, 3, 4, 5])
nservs = np.array([1, 2, 3, 4, 5])
tmaxs = [100, 500, 1000, 2000, 5000, 10000, 15000, 20000]
#tmaxs = [100, 500, 1000, 2000]

rr_mxm1 = np.array([0.1, 0.2, 0.3]) # rates ratio: lambda / mu
rr_mmx1 = np.array([0.7, 0.8, 0.9]) # rates ratio: lambda / mu
rr_mmc = np.array([0.7, 0.8, 0.9]) # rates ratio: lambda / mu

lambdas = np.repeat(1.0, 3) # lambda can always be taken as 1
mus_mxm1 = lambdas / rr_mxm1
mus_mmx1 = lambdas / rr_mmx1
mus_mmc = lambdas / rr_mmc 
rep_mc = 100

### Simulating Mx-M-1

In [5]:
sim_MxM1 = pd.DataFrame()

for batch in arr_batchs:
    sim = tab_mc_queue(
        lambdas = lambdas,
        mus = mus_mxm1,
        tmaxs = tmaxs,
        farr = lambda lmbda: lambda: np.random.exponential(1/lmbda),
        fdep = lambda mu: lambda: np.random.exponential(1/mu),
        nserv = 1,
        farr_batch = lambda: batch,
        fdep_batch = lambda: 1,
        rep_mc = rep_mc,
    )

    sim["batch"] = batch
    sim_MxM1 = pd.concat([sim_MxM1, sim], ignore_index=True)

100.0%

In [ ]:
sim_MxM1[sim_MxM1["batch"] == 3].head(10)

,lambda,mu,tmax,Wq_mean,Lq_mean,Rho_mean,Wq_var,Lq_var,Rho_var,batch
48,1.0,10.000000,100,0.180976,0.548266,0.297079,0.001125,0.019285,0.001340,3
49,1.0,10.000000,500,0.186462,0.559918,0.298900,0.000317,0.004423,0.000208,3
50,1.0,10.000000,1000,0.186288,0.560112,0.300125,0.000152,0.002289,0.000124,3
51,1.0,10.000000,2000,0.186132,0.557499,0.299308,0.000065,0.000962,0.000060,3
52,1.0,10.000000,5000,0.185731,0.557340,0.300216,0.000025,0.000348,0.000026,3
53,1.0,10.000000,10000,0.185464,0.557035,0.300333,0.000012,0.000190,0.000012,3
54,1.0,10.000000,15000,0.185556,0.556757,0.299950,0.000008,0.000111,0.000009,3
55,1.0,10.000000,20000,0.185551,0.556741,0.299987,0.000006,0.000082,0.000006,3
56,1.0,5.000000,100,0.744642,2.250163,0.587080,0.050712,0.689430,0.004373,3
57,1.0,5.000000,500,0.780771,2.353187,0.598109,0.015831,0.203432,0.000945,3


### Simulating M-Mx-1

In [ ]:
sim_MMx1 = pd.DataFrame()

for batch in dep_batchs:
    sim = tab_mc_queue(
        lambdas = lambdas,
        mus = mus_mmx1,
        tmaxs = tmaxs,
        farr = lambda lmbda: lambda: np.random.exponential(1/lmbda), # so we have a function for each lambda
        fdep = lambda mu: lambda: np.random.exponential(1/mu),
        nserv = 1,
        farr_batch = lambda: 1,
        fdep_batch = lambda: batch,
        rep_mc = rep_mc,
    )
    sim["batch"] = batch
    sim_MMx1 = pd.concat([sim_MMx1, sim], ignore_index=True)

20.8%

In [ ]:
sim_MMx1[sim_MMx1["batch"] == 3].head(10)

### Simulating M-M-c

In [ ]:
sim_MMc = pd.DataFrame()

for nserv in nservs:
    sim = tab_mc_queue(
        lambdas = lambdas,
        mus = mus_mmc,
        tmaxs = tmaxs,
        farr = lambda lmbda: lambda: np.random.exponential(1/lmbda),
        fdep = lambda mu: lambda: np.random.exponential(1/mu),
        nserv = nserv,
        farr_batch = lambda: 1,
        fdep_batch = lambda: 1,
        rep_mc = rep_mc,
    )

    sim["nserv"] = nserv
    sim_MMc = pd.concat([sim_MMc, sim], ignore_index=True)

In [ ]:
sim_MMc[sim_MMc["nserv"] == 3].head(10)

### Save results and clear variables

In [ ]:
filename = f"sim_{date.today()}.pkl"

results = {
    'lambdas': lambdas,
    'rr_mxm1': mus_mxm1,
    'rr_mmx1': mus_mmx1,
    'rr_mmc': mus_mmc,

    'lambdas': lambdas,
    'mus_mxm1': mus_mxm1,
    'mus_mmx1': mus_mmx1,
    'mus_mmc': mus_mmc,

    'arr_batch': arr_batchs,
    'dep_batch': dep_batchs,
    'nservs': nservs,
    'tmaxs': tmaxs,
    'rep_mc': rep_mc,
    'sim_MxM1': sim_MxM1,
    'sim_MMx1': sim_MMx1,
    'sim_MMc': sim_MMc,
}

with open(filename, 'wb') as f:
    pickle.dump(results, f)

del rr_mxm1, rr_mmx1, rr_mmc, lambdas, mus_mxm1, mus_mmx1, mus_mmc
del tmaxs, rep_mc, arr_batchs, dep_batchs, filename, results

## Testing queue generator
---

### M-M-c test

Theoretical metrics

In [ ]:
def mmc_metrics(lmbda, mu, c):
    rho = lmbda / (c * mu)
    s   = sum((lmbda / mu)**n / math.factorial(n) for n in range(c))
    t   = (lmbda / mu)**c / (math.factorial(c) * (1 - rho))
    P0  = 1 / (s + t)
    Pw  = t * P0
    Wq  = Pw / (c * mu - lmbda)
    W   = Wq + 1 / mu
    Lq  = lmbda * Wq
    L   = lmbda * W
    
    return {
        'rho': rho,
        'Pw': Pw,
        'Wq': Wq,
        'W': W,
        'Lq': Lq,
        'L': L
    }

In [ ]:
nserv  = 3
rho    = 0.5
lmbda  = 2.0
mu     = lmbda / (nserv * rho)

rho, Pw, Wq, W, Lq, L = mmc_metrics(lmbda, mu, nserv)

print("Rho:", round(rho, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))
print("Lq :", round(Lq, 4))
print("L  :", round(L, 4))
print("Pw :", round(Pw, 4))

Simulated metrics

In [ ]:
farr = lambda: rnd.expovariate(lmbda)
fdep = lambda: rnd.expovariate(mu)

q = Queue(nserv=nserv, farr=farr, fdep=fdep, tmax=10000)

q.run()
q.print_metrics()

### Mx-M-1 test

Theoretical metrics

In [ ]:
def mxm1_metrics(lmbda, batch_size, mu_global):
    lmbda_global = lmbda * batch_size
    rho       = lmbda_global / mu_global
    EX2       = batch_size**2
    L         = (rho + (lmbda / mu_global) * EX2) / (2 * (1 - rho))
    W         = L / lmbda_global
    Wq        = W - 1 / mu_global
    Lq        = lmbda_global * Wq
    return {
        'lambda_global': float(lmbda_global),
        'rho': float(rho),
        'L': float(L),
        'Lq': float(Lq),
        'W': float(W),
        'Wq': float(Wq)
    }

In [ ]:
lmbda_val = 0.2
batch     = 5
mu_val    = 2.0

lmbda_eff, rho, L, Lq, W, Wq = mxm1_metrics(lmbda_val, batch, mu_val)

print("rho       :", round(rho, 4))
print("L         :", round(L, 4))
print("Lq        :", round(Lq, 4))
print("W         :", round(W, 4))
print("Wq        :", round(Wq, 4))
print("lambda_eff:", round(lmbda_eff, 4))

Simulated metrics

In [ ]:
farr = lambda: rnd.expovariate(lmbda_val)
fdep = lambda: rnd.expovariate(mu_val)
farr_batch=lambda: batch
fdep_batch=lambda: 1

q = Queue(nserv=1, farr=farr, fdep=fdep, farr_batch=farr_batch, fdep_batch=fdep_batch, tmax=100000)

q.run()
q.print_metrics()

### M-G-1 test

Theoretical metrics

In [ ]:
def mg1_metrics(lmbda, ES, VarS):
    rho = lmbda * ES
    Cv2 = VarS / (ES**2)

    Lq  = ((1 + Cv2) / 2) * (rho**2 / (1 - rho))
    W   = ((1 + Cv2) / 2) * (rho / (1/ES - lmbda)) + ES
    L   = Lq + rho
    Wq  = W - ES

    return rho, L, Lq, W, Wq

lmbda = 0.8
a, b  = 0.7, 1.7

ES    = (a + b) / 2
VarS  = (b - a)**2 / 12

rho, L, Lq, W, Wq = mg1_metrics(lmbda, ES, VarS)

print("rho:", round(rho, 4))
print("L  :", round(L, 4))
print("Lq :", round(Lq, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))

In [ ]:
(L, 4))
print("Lq :", round(Lq, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))

Simulated metrics

In [ ]:
q = Queue(
    farr = lambda: rnd.expovariate(lmbda),
    fdep = lambda: rnd.uniform(a, b),
    tmax = 200000
)

q.run()
q.print_metrics()